# Cursul 6. Analiza exploratorie
**Fluxuri de lucru pentru știința datelor · Sociologie · 2026**

Datele sunt curate (C4) și în baza de date (C5). Azi întrebăm: **ce tipare vedem și ce explicații merită verificate?** Pornim de la ipoteze din literatură, scriem ce ne așteptăm să vedem, apoi ne uităm la date. În a doua parte privim aceleași perechi împreună, ca **rețea**.

| Ipoteza | Ce ne așteptăm | Cum verificăm |
|---|---|---|
| **H1. Distanța** | SCI scade cu distanța | nor de puncte, corelație, mediane pe intervale |
| **H2. Granița** | două regiuni din aceeași țară sunt mai legate, la aceeași distanță | mediane pe intervale, aceeași țară față de țări diferite |
| **H3. Economia** | regiunile asemănătoare economic sunt mai legate | corelația dintre decalajul de PIB și SCI |
| **H4. Oamenii care circulă** | țările între care trăiesc mulți migranți sau zboară mulți pasageri sunt mai legate | nor de puncte și corelație, pe perechi de țări |

[Bailey și colaboratorii (2020)](https://doi.org/10.1007/978-3-030-60975-7_1) găsesc pentru Europa efecte puternice ale distanței și granițelor; [Bailey și colaboratorii (2018)](https://doi.org/10.1257/jep.32.3.259) arată că SCI e asociat cu migrația.

**Ce înseamnă „verificăm” aici:** căutăm tipare compatibile sau nu cu așteptarea. Nu demonstrăm cauze.

| Pas | Ce facem | Timp |
|---|---|---|
| 1 | Datele, din baza de date | 5 min |
| 2 | Cum arată o pereche obișnuită; harta | 10 min |
| 3 | H1 și H2: distanța și granița | 20 min |
| 4 | H3 și H4: economia, migrația, zborurile | 20 min |
| 5 | Aceleași date, ca rețea: partenerii și comunitățile | 20 min |

## 1. Datele, din baza de date
Ne conectăm ca în C5 și citim cele trei tabele. `select *` înseamnă toate coloanele.

In [ ]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import networkx as nx
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv()
engine = create_engine(os.getenv("DATABASE_URL"))

regiuni = pd.read_sql("select * from regiuni", engine)
perechi = pd.read_sql("select * from perechi_regiuni", engine)
tari = pd.read_sql("select * from perechi_tari", engine)
print(len(regiuni), "regiuni ·", len(perechi), "perechi de regiuni ·", len(tari), "perechi de țări")

## 2. Cum arată o pereche obișnuită?
Înainte de orice comparație ne uităm la distribuție. **Anticipează:** media va fi mai mare sau mai mică decât mediana?

In [ ]:
perechi["sci"].describe().round(0)

In [ ]:
fig, axe = plt.subplots(1, 2, figsize=(11, 4))
perechi["sci"].hist(bins=40, ax=axe[0])
np.log10(perechi["sci"]).hist(bins=40, ax=axe[1])
axe[0].set(title="SCI, scara originală", xlabel="SCI")
axe[1].set(title="Aceleași date, scara log10", xlabel="log10(SCI)");

Câteva perechi foarte legate trag media în sus. Pe scara logaritmică (2 înseamnă 100, 3 înseamnă 1.000) distribuția se vede. De aici înainte folosim scara log pentru SCI.

### Harta: și lipsurile trebuie să se vadă
Contururile regiunilor vin de la GISCO, ca nomenclatorul din C3; `gpd.read_file` le citește direct de la adresă. Lipim indicatorii după cod, ca în C4.

In [ ]:
harta = gpd.read_file("https://gisco-services.ec.europa.eu/distribution/v2/nuts/geojson/NUTS_RG_20M_2024_3035_LEVL_2.geojson")
harta = harta[["NUTS_ID", "geometry"]].merge(regiuni, left_on="NUTS_ID", right_on="regiune")

ax = harta.plot(column="pib_locuitor", legend=True, figsize=(9, 7), missing_kwds={"color": "lightgrey"})
ax.set(xlim=(2.5e6, 7.4e6), ylim=(1.4e6, 5.5e6), title="PIB pe locuitor (PPS), gri = lipsă")
ax.set_axis_off()

**Interpretează:** un contrast și o limită a hărții. Regiunile mari ocupă mult loc pe hartă, fără să aibă neapărat mulți locuitori.

## 3. H1 și H2: distanța și granița
### 3.1. H1: legăturile scad cu distanța?
Avem nevoie de distanța dintre regiuni: luăm centrul fiecărei regiuni. Harta e în proiecția europeană EPSG:3035, care măsoară în metri, deci distanța dintre două centre se calculează direct. Centrul geometric nu e capitala și nici centrul populației, dar e o aproximare bună pentru explorare.

In [ ]:
centre = harta.set_index("regiune").centroid           # centrul fiecărei regiuni

capat_1 = centre[perechi["regiune_1"]].values           # centrul primului capăt al fiecărei perechi
capat_2 = centre[perechi["regiune_2"]].values           # centrul celui de-al doilea
perechi["distanta_km"] = capat_1.distance(capat_2) / 1000
perechi[["regiune_1", "regiune_2", "distanta_km", "sci"]].head()

In [ ]:
puncte = perechi.sample(5000, random_state=42)          # 5.000 de perechi, ca punctele să nu se acopere
ax = puncte.plot.scatter(x="distanta_km", y="sci", logx=True, logy=True, alpha=0.15, s=6)
ax.set(xlabel="Distanța (km, log)", ylabel="SCI (log)", title="H1: 5.000 de perechi");

Desenăm doar 5.000 de perechi; calculele folosesc toate perechile. **Spearman** compară ordinea valorilor și e între -1 și 1: negativ înseamnă că perechile mai îndepărtate tind să aibă SCI mai mic.

In [ ]:
perechi["distanta_km"].corr(perechi["sci"], method="spearman")

Mediana SCI pe intervale de distanță arată forma relației mai bine decât un singur număr. `pd.cut` pune fiecare distanță în intervalul ei; apoi agregăm, ca în C5.

In [ ]:
praguri = [0, 100, 250, 500, 1000, 2000, 10000]
perechi["interval_km"] = pd.cut(perechi["distanta_km"], praguri)
perechi.groupby("interval_km", observed=True)["sci"].agg(["size", "median"])

### 3.2. H2: aceeași țară, legături mai puternice?
În C5 am văzut că SCI-ul mediu e mult mai mare în aceeași țară. Dar perechile interne sunt și mai apropiate, deci amestecăm granița cu distanța. De aceea comparăm **în același interval de distanță**.

In [ ]:
perechi["aceeasi_tara"] = perechi["tara_1"] == perechi["tara_2"]
h2 = perechi.groupby(["interval_km", "aceeasi_tara"], observed=True)["sci"].median().unstack()   # intervale × (da, nu)
h2

In [ ]:
ax = h2.plot.bar(logy=True, rot=30)
ax.set(xlabel="Distanța (km)", ylabel="SCI median (log)", title="H2: la aceeași distanță");

**Interpretează:** de câte ori e mai mare mediana internă decât cea externă, la 250-500 km? Se păstrează diferența la toate distanțele? Nu e încă efectul cauzal al graniței: limba și istoria rămân amestecate.

## 4. H3 și H4: economia, migrația, zborurile
### 4.1. H3: asemănarea economică
Perechea nu are sens, deci folosim **diferența absolută** de PIB. Lucrăm doar cu perechile care au PIB la ambele capete, și spunem câte sunt.

In [ ]:
economice = perechi.dropna(subset=["pib_locuitor_1", "pib_locuitor_2"]).copy()   # PIB la ambele capete
economice["decalaj_pib"] = (economice["pib_locuitor_1"] - economice["pib_locuitor_2"]).abs()
print(len(economice), "perechi cu PIB, din", len(perechi))
economice["decalaj_pib"].corr(economice["sci"], method="spearman")

**Interpretează:** semnul e cel așteptat? Bailey și colaboratorii (2020) găsesc că decalajele mai mari pot merge cu legături mai puternice, iar migrația e o explicație posibilă: oamenii pleacă din regiunile mai sărace spre cele mai bogate. Asta ne duce la H4.

### 4.2. H4: migrația
Trecem la perechile de țări, cele cu migrația cunoscută în ambele sensuri.

In [ ]:
cu_migratie = tari.dropna(subset=["migranti"])
print(len(cu_migratie), "perechi de țări")

ax = cu_migratie.plot.scatter(x="migranti", y="sci", logx=True, logy=True, alpha=0.6)
ax.set(xlabel="Migranți în ambele sensuri (log)", ylabel="SCI (log)", title="H4: migrație și legături sociale")
cu_migratie["migranti"].corr(cu_migratie["sci"], method="spearman")

Corelația e pozitivă, dar nu foarte mare. Un motiv: SCI e **relativ**, împărțit la numărul de utilizatori din cele două țări. Un milion de migranți înseamnă mult pentru Austria și puțin pentru Germania. Comparăm atunci cu o **rată**: migranți la 1.000 de locuitori ai celor două țări, cu populația de la Banca Mondială, din C2.

In [ ]:
bm = pd.read_csv("data/raw/bm.csv", keep_default_na=False, na_values=[""])
populatie = bm[(bm["indicator"] == "populatie") & (bm["an"] == 2024)].set_index("tara")["valoare"]   # țară -> populație

cu_migratie["populatie"] = cu_migratie["tara_1"].map(populatie) + cu_migratie["tara_2"].map(populatie)
cu_migratie["la_mie"] = 1000 * cu_migratie["migranti"] / cu_migratie["populatie"]
cu_migratie["la_mie"].corr(cu_migratie["sci"], method="spearman")

**Interpretează:** cât crește corelația? Ce ne spune asta despre felul în care e construit SCI?

### 4.3. Zborurile
Migrația spune câți oameni s-au mutat; zborurile, cât se circulă. **Exercițiu.** Corelația Spearman dintre `pasageri` și `sci`, pe perechile care au zboruri. Comparativ cu migrația, e mai mare sau mai mică?

In [ ]:
# 1. Perechile de țări care au pasageri (dropna pe coloana pasageri)
# 2. Corelația Spearman dintre pasageri și sci, ca la migrație

**Ghicitoarea din C1.** Perechile României, după SCI, cu migranții și rata lor:

In [ ]:
cu_ro = cu_migratie[(cu_migratie["tara_1"] == "RO") | (cu_migratie["tara_2"] == "RO")]
cu_ro.sort_values("sci", ascending=False)[["tara_1", "tara_2", "sci", "migranti", "la_mie", "pasageri"]].head(8)

Italia are cea mai mare rată de migranți, dar Austria, Germania, Ungaria și Irlanda au SCI mai mare. Migrația explică mult, dar nu tot. Ungaria e un indiciu: puțini migranți, dar SCI mare, pentru că o parte din legături vin din minoritatea maghiară din România, nu din migrație. În C7 punem distanța și migrația în același model și vedem ce rămâne.

## 5. Aceleași date, ca rețea
Până acum am privit perechile una câte una. O **rețea** le privește împreună: fiecare regiune e un **nod**, fiecare pereche o **legătură**, cu SCI ca **pondere**.

In [ ]:
G = nx.from_pandas_edgelist(perechi, "regiune_1", "regiune_2", edge_attr="sci")
print(G.number_of_nodes(), "noduri ·", G.number_of_edges(), "legături")

Fiecare regiune e legată de toate celelalte, deci numărul de legături nu spune nimic. Spune însă **forța**: suma ponderilor legăturilor unei regiuni.

In [ ]:
regiuni["forta"] = regiuni["regiune"].map(dict(G.degree(weight="sci")))
regiuni.sort_values("forta", ascending=False)[["regiune", "denumire", "tara", "forta"]].head(10)

Forța e o sumă de indici relativi, nu un număr de prieteni: regiunile mici apar sus pentru că SCI e împărțit la numărul de utilizatori, aceeași capcană ca la migrație.

### 5.1. Partenerii unei regiuni
Alegem o regiune și cei mai puternici 10 parteneri ai ei **din alte țări**. Regiunea poate fi pe oricare capăt al perechii, deci luăm ambele cazuri și le punem unul sub altul.

In [ ]:
REGIUNE = "RO41"        # Sud-Vest Oltenia; încearcă și "RO11" (Nord-Vest)

ca_primul = perechi[perechi["regiune_1"] == REGIUNE][["regiune_2", "tara_2", "sci"]]   # regiunea e capătul 1
ca_al_doilea = perechi[perechi["regiune_2"] == REGIUNE][["regiune_1", "tara_1", "sci"]]  # regiunea e capătul 2
ca_primul.columns = ["partener", "tara", "sci"]
ca_al_doilea.columns = ["partener", "tara", "sci"]

legaturi = pd.concat([ca_primul, ca_al_doilea])
externi = legaturi[legaturi["tara"] != REGIUNE[:2]].nlargest(10, "sci")   # alte țări, primii 10
externi

**Anticipează:** în ce țări vor fi partenerii? Notează înainte de hartă. Liniile unesc centrele regiunilor; nu sunt drumuri.

In [ ]:
ax = harta.plot(color="#eeeeee", edgecolor="white", figsize=(9, 7))
for partener in externi["partener"]:
    ax.plot([centre[REGIUNE].x, centre[partener].x], [centre[REGIUNE].y, centre[partener].y], color="#c0392b")
harta[harta["regiune"] == REGIUNE].plot(ax=ax, color="#c0392b")
ax.set(xlim=(2.5e6, 7.4e6), ylim=(1.4e6, 5.5e6), title=f"Primii 10 parteneri externi ai {REGIUNE}")
ax.set_axis_off()

### 5.2. Comunitățile
O **comunitate** e un grup de regiuni mai legate între ele decât cu restul. Algoritmul [Louvain](https://networkx.org/documentation/stable/reference/algorithms/generated/networkx.algorithms.community.louvain.louvain_communities.html) le caută doar din ponderi: nu știe nimic despre țări sau hartă. `seed` fixează punctul de pornire, ca rezultatul să fie același la fiecare rulare.

In [ ]:
grupuri = nx.community.louvain_communities(G, weight="sci", seed=1)   # o listă de grupuri de regiuni
print(len(grupuri), "comunități")

comunitate = {}
for numar, grup in enumerate(grupuri):
    for regiune in grup:
        comunitate[regiune] = numar          # regiune -> numărul grupului ei
harta["comunitate"] = harta["regiune"].map(comunitate)

In [ ]:
ax = harta.plot(column="comunitate", categorical=True, cmap="tab20", figsize=(9, 7))
ax.set(xlim=(2.5e6, 7.4e6), ylim=(1.4e6, 5.5e6), title="Comunitățile rețelei SCI")
ax.set_axis_off()

Culorile sunt doar etichete: comunitatea 3 nu e „mai mult” decât comunitatea 2. **Interpretează:** comunitățile urmează granițele? Găsește una care trece de o graniță: ce ar putea explica asta (migrație, limbă, istorie)? Gruparea singură nu spune care; pentru asta avem nevoie de alte date.

## Fișa echipei
| Ce raportați | Din rulare |
|---|---|
| H1-H4: ce ați găsit (un număr, cu câte perechi) | ... |
| Compatibile, incompatibile sau mixte cu așteptarea? | ... |
| Partenerii externi ai unei regiuni din coridorul vostru | ... |
| O explicație alternativă și o limită | ... |

**Exercițiul echipei.** Perechea coridorului vostru, în `cu_migratie`: SCI, migranții, rata și pasagerii. E peste sau sub perechile cu o rată de migrație asemănătoare?

In [ ]:
# Perechea coridorului vostru în cu_migratie (tara_1 și tara_2, în ordine alfabetică)

## Ce ducem mai departe
Avem patru ipoteze verificate exploratoriu și o primă privire asupra rețelei. În C7 construim variabilele (distanța, granița, rata migranților) și le punem în același model.

**Poți explica fără cod?** De ce folosim scara logaritmică pentru SCI? Ce măsoară Spearman? De ce comparăm granița la aceeași distanță? De ce o rată de migranți e mai potrivită decât un număr? Ce e forța unui nod și ce e o comunitate?